## Objective

The objective of this notebook is to use the selected forecasting model to support stock recommendation decisions.

In this step, the final model will be used to estimate future product demand and help identify which products may need stock replenishment.

The main goals are:

* Load the selected trained model.
* Generate demand predictions for each product.
* Compare predicted demand with available stock information.
* Create simple stock recommendation rules.
* Identify products with possible stockout risk.
* Support inventory decisions using the forecasting results.

This step connects the machine learning model with the business problem. Instead of only predicting sales, the goal is to transform the predictions into useful recommendations for stock management.

In [1]:
import os 
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

ROOT = Path("..").resolve()
os.chdir(ROOT)
sys.path.append(str(ROOT))

DATA_PATH = Path("data/processed/modeling_dataset.csv")
EVAL_PATH = Path("reports/model_evaluation_summary.csv")
EVAL_PROD_PATH = Path("reports/model_evaluation_by_product.csv")
MODELS_PATH = ROOT / "artifacts" / "models"
STOCK_PATH = ROOT / "data" / "processed" / "daily_product_sales.csv"
from src.recommendation import generate_stock_recommendations

model = joblib.load(MODELS_PATH / "xgboost_model.pkl")
df = pd.read_csv(DATA_PATH)
evaluation_summary = pd.read_csv(EVAL_PATH)
evaluation_product_summary = pd.read_csv(EVAL_PROD_PATH)
stock_df =  pd.read_csv(STOCK_PATH)
stock_df["sale_date"] = pd.to_datetime(stock_df["sale_date"])
forecast_horizon_days = 14

## Get Latest Product State

In [2]:
latest_product_state = (
    df
    .sort_values(["product_name", "sale_date"])
    .groupby("product_name", as_index=False)
    .tail(1)
    .reset_index(drop=True)
    .rename(columns={"sale_date": "latest_sale_date"})
)

latest_product_state.head()

,product_name,latest_sale_date,quantity_sold,unit_price_brl,day_of_week,day_of_month,month,week_of_year,is_weekend,quantity_1d,quantity_7d,quantity_14d,rolling_1d,rolling_7d,rolling_14d,weather_condition
0,Bag Delivery 45L,2026-05-29,3.0,181.016667,4,29,5,22,0,4.0,10.0,2.0,4.0,4.857143,4.071429,Chuva Forte
1,Bag Delivery 80L,2026-05-29,1.0,268.030000,4,29,5,22,0,1.0,3.0,2.0,1.0,1.571429,1.571429,Chuva Forte
2,Balaclava,2026-05-29,0.0,35.040000,4,29,5,22,0,0.0,0.0,1.0,0.0,0.285714,0.428571,Chuva Forte
3,Baú Moto 30L,2026-05-29,1.0,253.510000,4,29,5,22,0,6.0,1.0,0.0,6.0,1.571429,1.500000,Chuva Forte
4,Capa de Chuva,2026-05-29,10.0,99.755000,4,29,5,22,0,3.0,2.0,0.0,3.0,2.714286,2.214286,Chuva Forte


## Create Future Dates

In [3]:
df["sale_date"] = pd.to_datetime(df["sale_date"])

last_historical_date = df["sale_date"].max()

future_dates = pd.date_range(
    start=last_historical_date + pd.Timedelta(days=1),
    periods=forecast_horizon_days,
    freq="D"
)

future_dates_df = pd.DataFrame({
    "sale_date": future_dates
})


future_dates_df.head()

,sale_date
0,2026-05-30
1,2026-05-31
2,2026-06-01
3,2026-06-02
4,2026-06-03


## Build Future Feature Dataset

In [4]:
future_features_df = latest_product_state.merge(
    future_dates_df,
    how="cross"
)

# Calendar features must describe each future date, not the last historical row.
future_features_df["sale_date"] = pd.to_datetime(future_features_df["sale_date"])
future_features_df["day_of_week"] = future_features_df["sale_date"].dt.dayofweek
future_features_df["day_of_month"] = future_features_df["sale_date"].dt.day
future_features_df["month"] = future_features_df["sale_date"].dt.month
future_features_df["week_of_year"] = (
    future_features_df["sale_date"].dt.isocalendar().week.astype(int)
)
future_features_df["is_weekend"] = (
    future_features_df["day_of_week"].isin([5, 6]).astype(int)
)

model_feature_cols = [
    "product_name", "unit_price_brl", "day_of_week", "day_of_month",
    "month", "week_of_year", "is_weekend", "quantity_1d", "quantity_7d",
    "quantity_14d", "rolling_1d", "rolling_7d", "rolling_14d",
    "weather_condition"
]
future_X = future_features_df[model_feature_cols].copy()

expected_rows = latest_product_state["product_name"].nunique() * forecast_horizon_days
assert len(future_features_df) == expected_rows
assert not future_features_df.duplicated(["product_name", "sale_date"]).any()
assert (future_features_df.loc[future_features_df["day_of_week"].isin([5, 6]), "is_weekend"] == 1).all()
assert (future_features_df["month"] == future_features_df["sale_date"].dt.month).all()
assert (future_features_df["week_of_year"] == future_features_df["sale_date"].dt.isocalendar().week.astype(int)).all()

future_features_df[["product_name", "sale_date", "day_of_week", "day_of_month", "month", "week_of_year", "is_weekend"]].head(14)


,product_name,sale_date,day_of_week,day_of_month,month,week_of_year,is_weekend
0,Bag Delivery 45L,2026-05-30,5,30,5,22,1
1,Bag Delivery 45L,2026-05-31,6,31,5,22,1
2,Bag Delivery 45L,2026-06-01,0,1,6,23,0
3,Bag Delivery 45L,2026-06-02,1,2,6,23,0
4,Bag Delivery 45L,2026-06-03,2,3,6,23,0
5,Bag Delivery 45L,2026-06-04,3,4,6,23,0
6,Bag Delivery 45L,2026-06-05,4,5,6,23,0
7,Bag Delivery 45L,2026-06-06,5,6,6,23,1
8,Bag Delivery 45L,2026-06-07,6,7,6,23,1
9,Bag Delivery 45L,2026-06-08,0,8,6,24,0


## Generate Demand Forecast

In [5]:
forecast_raw = model.predict(future_X)

forecast_non_negative = np.clip(
    forecast_raw,
    0,
    None
)

future_forecast_df = future_features_df[
    [
        "product_name",
        "sale_date",
        "unit_price_brl",
        "weather_condition"
    ]
].copy()

future_forecast_df["forecast_raw"] = forecast_raw
future_forecast_df["forecast_non_negative"] = forecast_non_negative

forecast_summary = pd.DataFrame({
    "metric": [
        "Lowest raw forecast",
        "Highest raw forecast",
        "Negative raw forecasts",
        "Lowest non-negative forecast",
        "Highest non-negative forecast"
    ],
    "value": [
        future_forecast_df["forecast_raw"].min(),
        future_forecast_df["forecast_raw"].max(),
        (future_forecast_df["forecast_raw"] < 0).sum(),
        future_forecast_df["forecast_non_negative"].min(),
        future_forecast_df["forecast_non_negative"].max()
    ]
})

forecast_summary

,metric,value
0,Lowest raw forecast,0.125105
1,Highest raw forecast,4.917656
2,Negative raw forecasts,0.000000
3,Lowest non-negative forecast,0.125105
4,Highest non-negative forecast,4.917656


## Aggregate Forecast by Product

In [6]:
forecast_by_product_df = (
    future_forecast_df
    .groupby("product_name", as_index=False)
    .agg(
        forecast_horizon_days=("sale_date", "nunique"),
        forecasted_demand_raw=("forecast_raw", "sum"),
        forecasted_demand_non_negative=("forecast_non_negative", "sum")
    )
)

# Aggregate daily forecasts first and round only the product total.
forecast_by_product_df["forecasted_demand_units"] = (
    np.ceil(forecast_by_product_df["forecasted_demand_non_negative"]).astype(int)
)

assert (forecast_by_product_df["forecast_horizon_days"] == forecast_horizon_days).all()
assert (forecast_by_product_df["forecasted_demand_units"] >= 0).all()
forecast_by_product_df.head()


,product_name,forecast_horizon_days,forecasted_demand_raw,forecasted_demand_non_negative,forecasted_demand_units
0,Bag Delivery 45L,14,36.496861,36.496861,37
1,Bag Delivery 80L,14,28.752541,28.752541,29
2,Balaclava,14,17.978758,17.978758,18
3,Baú Moto 30L,14,28.651274,28.651274,29
4,Capa de Chuva,14,44.332775,44.332775,45


## Add Current Stock and Lead Time

In [7]:
valid_stock_df = stock_df.dropna(subset=["current_stock_snapshot"])

current_stock_df = (
    valid_stock_df
    .sort_values(["product_name", "sale_date"])
    .groupby("product_name")
    .tail(1)[["product_name", "current_stock_snapshot"]]
    .reset_index(drop=True)
    .rename(columns={"current_stock_snapshot": "current_stock"})
)

forecast_products = set(forecast_by_product_df["product_name"])
valid_stock_products = set(current_stock_df["product_name"])
products_without_valid_snapshot = sorted(forecast_products - valid_stock_products)

print("Products without a valid stock snapshot:", len(products_without_valid_snapshot))
print("Products filled with zero:", products_without_valid_snapshot)
print("Products with a valid stock snapshot:", len(forecast_products & valid_stock_products))
current_stock_df.head()


Products without a valid stock snapshot: 0
Products filled with zero: []
Products with a valid stock snapshot: 12


,product_name,current_stock
0,Bag Delivery 45L,68.00
1,Bag Delivery 80L,108.00
2,Balaclava,59.00
3,Baú Moto 30L,63.00
4,Capa de Chuva,64.25


In [8]:
default_lead_time_days = 7

lead_time_df = forecast_by_product_df[["product_name"]].copy()
lead_time_df["supplier_lead_time_days"] = default_lead_time_days

recommendation_base_df = (
    forecast_by_product_df
    .merge(current_stock_df, on="product_name", how="left")
    .merge(lead_time_df, on="product_name", how="left")
)

recommendation_base_df["current_stock"] = (
    recommendation_base_df["current_stock"].fillna(0).clip(lower=0).round().astype(int)
)
recommendation_base_df["supplier_lead_time_days"] = (
    recommendation_base_df["supplier_lead_time_days"]
    .fillna(default_lead_time_days).clip(lower=1).astype(int)
)

assert recommendation_base_df["product_name"].is_unique
assert recommendation_base_df["current_stock"].notna().all()
recommendation_base_df.head()


,product_name,forecast_horizon_days,forecasted_demand_raw,forecasted_demand_non_negative,forecasted_demand_units,current_stock,supplier_lead_time_days
0,Bag Delivery 45L,14,36.496861,36.496861,37,68,7
1,Bag Delivery 80L,14,28.752541,28.752541,29,108,7
2,Balaclava,14,17.978758,17.978758,18,59,7
3,Baú Moto 30L,14,28.651274,28.651274,29,63,7
4,Capa de Chuva,14,44.332775,44.332775,45,64,7


In [9]:
recommendation_base_df[
    [
        "product_name",
        "forecasted_demand_units",
        "current_stock",
        "supplier_lead_time_days"
    ]
].head()

,product_name,forecasted_demand_units,current_stock,supplier_lead_time_days
0,Bag Delivery 45L,37,68,7
1,Bag Delivery 80L,29,108,7
2,Balaclava,18,59,7
3,Baú Moto 30L,29,63,7
4,Capa de Chuva,45,64,7


## Define Safety Stock

In [10]:
safety_stock_pct = 0.20

recommendation_base_df["safety_stock"] = (
    np.ceil(
        recommendation_base_df["forecasted_demand_units"] * safety_stock_pct
    )
    .astype(int)
)

recommendation_base_df.head()

,product_name,forecast_horizon_days,forecasted_demand_raw,forecasted_demand_non_negative,forecasted_demand_units,current_stock,supplier_lead_time_days,safety_stock
0,Bag Delivery 45L,14,36.496861,36.496861,37,68,7,8
1,Bag Delivery 80L,14,28.752541,28.752541,29,108,7,6
2,Balaclava,14,17.978758,17.978758,18,59,7,4
3,Baú Moto 30L,14,28.651274,28.651274,29,63,7,6
4,Capa de Chuva,14,44.332775,44.332775,45,64,7,9


In [11]:
recommendation_base_df[
    [
        "product_name",
        "forecasted_demand_units",
        "current_stock",
        "supplier_lead_time_days",
        "safety_stock"
    ]
].head()

,product_name,forecasted_demand_units,current_stock,supplier_lead_time_days,safety_stock
0,Bag Delivery 45L,37,68,7,8
1,Bag Delivery 80L,29,108,7,6
2,Balaclava,18,59,7,4
3,Baú Moto 30L,29,63,7,6
4,Capa de Chuva,45,64,7,9


In [12]:
recommendation_base_df["safety_stock"] = np.where(
    recommendation_base_df["forecasted_demand_units"] > 0,
    np.ceil(recommendation_base_df["forecasted_demand_units"] * safety_stock_pct),
    0
).astype(int)

## Calculate Required Stock

In [13]:
recommendation_base_df["required_stock"] = (
    recommendation_base_df["forecasted_demand_units"]
    + recommendation_base_df["safety_stock"]
)

recommendation_base_df.head()

,product_name,forecast_horizon_days,forecasted_demand_raw,forecasted_demand_non_negative,forecasted_demand_units,current_stock,supplier_lead_time_days,safety_stock,required_stock
0,Bag Delivery 45L,14,36.496861,36.496861,37,68,7,8,45
1,Bag Delivery 80L,14,28.752541,28.752541,29,108,7,6,35
2,Balaclava,14,17.978758,17.978758,18,59,7,4,22
3,Baú Moto 30L,14,28.651274,28.651274,29,63,7,6,35
4,Capa de Chuva,14,44.332775,44.332775,45,64,7,9,54


In [14]:
recommendation_base_df[
    [
        "product_name",
        "forecasted_demand_units",
        "safety_stock",
        "required_stock",
        "current_stock",
        "supplier_lead_time_days"
    ]
].head()

,product_name,forecasted_demand_units,safety_stock,required_stock,current_stock,supplier_lead_time_days
0,Bag Delivery 45L,37,8,45,68,7
1,Bag Delivery 80L,29,6,35,108,7
2,Balaclava,18,4,22,59,7
3,Baú Moto 30L,29,6,35,63,7
4,Capa de Chuva,45,9,54,64,7


## Calculate Recommended Purchase Quantity

In [15]:
recommendation_base_df["recommended_purchase_quantity"] = (
    recommendation_base_df["required_stock"]
    - recommendation_base_df["current_stock"]
).clip(lower=0).astype(int)

recommendation_base_df[
    [
        "product_name",
        "forecasted_demand_units",
        "safety_stock",
        "required_stock",
        "current_stock",
        "recommended_purchase_quantity"
    ]
].head()

,product_name,forecasted_demand_units,safety_stock,required_stock,current_stock,recommended_purchase_quantity
0,Bag Delivery 45L,37,8,45,68,0
1,Bag Delivery 80L,29,6,35,108,0
2,Balaclava,18,4,22,59,0
3,Baú Moto 30L,29,6,35,63,0
4,Capa de Chuva,45,9,54,64,0


## Create Stock Status

In [16]:
overstock_multiplier = 1.50

conditions = [
    # Current stock cannot cover forecasted demand
    recommendation_base_df["current_stock"]
    < recommendation_base_df["forecasted_demand_units"],

    # Current stock covers demand, but not demand + safety stock
    (
        recommendation_base_df["current_stock"]
        >= recommendation_base_df["forecasted_demand_units"]
    )
    & (
        recommendation_base_df["current_stock"]
        < recommendation_base_df["required_stock"]
    ),

    # Current stock is much higher than required stock
    (
        recommendation_base_df["required_stock"] > 0
    )
    & (
        recommendation_base_df["current_stock"]
        > recommendation_base_df["required_stock"] * overstock_multiplier
    ),

    # No predicted demand, but there is stock available
    (
        recommendation_base_df["forecasted_demand_units"] == 0
    )
    & (
        recommendation_base_df["current_stock"] > 0
    )
]

statuses = [
    "critical",
    "warning",
    "overstock",
    "overstock"
]

recommendation_base_df["stock_status"] = np.select(
    conditions,
    statuses,
    default="healthy"
)

recommendation_base_df[
    [
        "product_name",
        "forecasted_demand_units",
        "safety_stock",
        "required_stock",
        "current_stock",
        "recommended_purchase_quantity",
        "stock_status"
    ]
].head()

,product_name,forecasted_demand_units,safety_stock,required_stock,current_stock,recommended_purchase_quantity,stock_status
0,Bag Delivery 45L,37,8,45,68,0,overstock
1,Bag Delivery 80L,29,6,35,108,0,overstock
2,Balaclava,18,4,22,59,0,overstock
3,Baú Moto 30L,29,6,35,63,0,overstock
4,Capa de Chuva,45,9,54,64,0,healthy


## Create Priority Score

In [17]:
risk_weights = {
    "critical": 1.50,
    "warning": 1.20,
    "healthy": 1.00,
    "overstock": 0.00
}

recommendation_base_df["risk_weight"] = (
    recommendation_base_df["stock_status"]
    .map(risk_weights)
    .fillna(1.00)
)

recommendation_base_df["priority_score_raw"] = (
    recommendation_base_df["recommended_purchase_quantity"]
    * recommendation_base_df["supplier_lead_time_days"]
    * recommendation_base_df["risk_weight"]
)

max_priority = recommendation_base_df["priority_score_raw"].max()

if max_priority > 0:
    recommendation_base_df["priority_score"] = (
        recommendation_base_df["priority_score_raw"]
        / max_priority
        * 100
    ).round().astype(int)
else:
    recommendation_base_df["priority_score"] = 0
    

recommendation_base_df = (
    recommendation_base_df
    .sort_values(
        ["priority_score", "recommended_purchase_quantity"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

recommendation_base_df[
    [
        "product_name",
        "stock_status",
        "forecasted_demand_units",
        "current_stock",
        "recommended_purchase_quantity",
        "supplier_lead_time_days",
        "priority_score"
    ]
].head(10)

,product_name,stock_status,forecasted_demand_units,current_stock,recommended_purchase_quantity,supplier_lead_time_days,priority_score
0,Bag Delivery 45L,overstock,37,68,0,7,0
1,Bag Delivery 80L,overstock,29,108,0,7,0
2,Balaclava,overstock,18,59,0,7,0
3,Baú Moto 30L,overstock,29,63,0,7,0
4,Capa de Chuva,healthy,45,64,0,7,0
5,Capacete LS2,overstock,6,38,0,7,0
6,Capacete Pro Tork,overstock,29,67,0,7,0
7,Carregador USB Moto,overstock,13,95,0,7,0
8,Intercomunicador,overstock,4,94,0,7,0
9,Luva Motoboy,healthy,50,86,0,7,0


## Sort Final Recommendations

In [18]:
final_recommendations_df = generate_stock_recommendations(
    forecast_by_product_df=forecast_by_product_df,
    current_stock_df=current_stock_df,
    lead_time_df=lead_time_df,
    default_lead_time_days=default_lead_time_days,
    safety_stock_pct=0.20,
    overstock_multiplier=1.50
)

final_recommendations_df.head(10)


,product_name,forecast_horizon_days,forecasted_demand_raw,forecasted_demand_non_negative,forecasted_demand_units,current_stock,safety_stock,required_stock,recommended_purchase_quantity,supplier_lead_time_days,stock_status,priority_score
0,Capa de Chuva,14,44.332775,44.332775,45,64,9,54,0,7,healthy,0
1,Luva Motoboy,14,49.497753,49.497753,50,86,10,60,0,7,healthy,0
2,Suporte Celular Moto,14,60.443356,60.443356,61,83,13,74,0,7,healthy,0
3,Bag Delivery 45L,14,36.496861,36.496861,37,68,8,45,0,7,overstock,0
4,Bag Delivery 80L,14,28.752541,28.752541,29,108,6,35,0,7,overstock,0
5,Balaclava,14,17.978758,17.978758,18,59,4,22,0,7,overstock,0
6,Baú Moto 30L,14,28.651274,28.651274,29,63,6,35,0,7,overstock,0
7,Capacete LS2,14,5.172674,5.172674,6,38,2,8,0,7,overstock,0
8,Capacete Pro Tork,14,28.527456,28.527456,29,67,6,35,0,7,overstock,0
9,Carregador USB Moto,14,12.630710,12.630710,13,95,3,16,0,7,overstock,0


## Save Stock Recommendations

In [19]:
reports_path = ROOT / "reports"
reports_path.mkdir(parents=True, exist_ok=True)

stock_recommendation_cols = [
    "product_name", "forecast_horizon_days", "forecasted_demand_raw",
    "forecasted_demand_non_negative", "forecasted_demand_units",
    "current_stock", "safety_stock", "required_stock",
    "recommended_purchase_quantity", "supplier_lead_time_days",
    "stock_status", "priority_score"
]
stock_recommendations_df = final_recommendations_df[stock_recommendation_cols].copy()

essential_cols = stock_recommendation_cols
assert stock_recommendations_df["product_name"].is_unique
assert (stock_recommendations_df["forecast_horizon_days"] == forecast_horizon_days).all()
assert stock_recommendations_df[essential_cols].notna().all().all()
for column in ["forecasted_demand_units", "current_stock", "recommended_purchase_quantity"]:
    assert (stock_recommendations_df[column] >= 0).all()
    assert pd.api.types.is_integer_dtype(stock_recommendations_df[column])

output_path = reports_path / "stock_recommendations.csv"
stock_recommendations_df.to_csv(output_path, index=False, encoding="utf-8")
print("Stock recommendations saved successfully:")
print(output_path.resolve())
print("Rows:", len(stock_recommendations_df))
print("Columns:", len(stock_recommendations_df.columns))
stock_recommendations_df.head(10)


Stock recommendations saved successfully:
C:\DEV\motostock-ai\reports\stock_recommendations.csv
Rows: 12
Columns: 12


,product_name,forecast_horizon_days,forecasted_demand_raw,forecasted_demand_non_negative,forecasted_demand_units,current_stock,safety_stock,required_stock,recommended_purchase_quantity,supplier_lead_time_days,stock_status,priority_score
0,Capa de Chuva,14,44.332775,44.332775,45,64,9,54,0,7,healthy,0
1,Luva Motoboy,14,49.497753,49.497753,50,86,10,60,0,7,healthy,0
2,Suporte Celular Moto,14,60.443356,60.443356,61,83,13,74,0,7,healthy,0
3,Bag Delivery 45L,14,36.496861,36.496861,37,68,8,45,0,7,overstock,0
4,Bag Delivery 80L,14,28.752541,28.752541,29,108,6,35,0,7,overstock,0
5,Balaclava,14,17.978758,17.978758,18,59,4,22,0,7,overstock,0
6,Baú Moto 30L,14,28.651274,28.651274,29,63,6,35,0,7,overstock,0
7,Capacete LS2,14,5.172674,5.172674,6,38,2,8,0,7,overstock,0
8,Capacete Pro Tork,14,28.527456,28.527456,29,67,6,35,0,7,overstock,0
9,Carregador USB Moto,14,12.630710,12.630710,13,95,3,16,0,7,overstock,0


## Save Recommendation Summary

In [20]:
reports_path = ROOT / "reports"
reports_path.mkdir(parents=True, exist_ok=True)

# Get the product with the highest priority
if stock_recommendations_df.empty:
    highest_priority_product = None
    forecast_horizon_days = 0
else:
    highest_priority_product = (
        stock_recommendations_df
        .sort_values("priority_score", ascending=False)
        .iloc[0]["product_name"]
    )

    forecast_horizon_days = int(
        stock_recommendations_df["forecast_horizon_days"].max()
    )

recommendation_summary_df = pd.DataFrame([
    {
        "total_products": stock_recommendations_df["product_name"].nunique(),

        "critical_products": (
            stock_recommendations_df["stock_status"] == "critical"
        ).sum(),

        "warning_products": (
            stock_recommendations_df["stock_status"] == "warning"
        ).sum(),

        "healthy_products": (
            stock_recommendations_df["stock_status"] == "healthy"
        ).sum(),

        "overstock_products": (
            stock_recommendations_df["stock_status"] == "overstock"
        ).sum(),

        "total_recommended_purchase_units": (
            stock_recommendations_df["recommended_purchase_quantity"].sum()
        ),

        "highest_priority_product": highest_priority_product,
        "forecast_horizon_days": forecast_horizon_days,
        "selected_model": "XGBoost"
    }
])

recommendation_summary_df

,total_products,critical_products,warning_products,healthy_products,overstock_products,total_recommended_purchase_units,highest_priority_product,forecast_horizon_days,selected_model
0,12,0,0,3,9,0,Capa de Chuva,14,XGBoost


In [21]:
summary_output_path = (
    reports_path / "stock_recommendation_summary.csv"
)

recommendation_summary_df.to_csv(
    summary_output_path,
    index=False,
    encoding="utf-8"
)

print("Recommendation summary saved successfully:")
print(summary_output_path.resolve())

Recommendation summary saved successfully:
C:\DEV\motostock-ai\reports\stock_recommendation_summary.csv


## Create Business Findings

In [22]:
products_to_restock = (
    stock_recommendations_df[
        stock_recommendations_df["recommended_purchase_quantity"] > 0
    ]
    .sort_values("priority_score", ascending=False)
)

critical_products = stock_recommendations_df[
    stock_recommendations_df["stock_status"] == "critical"
]

highest_priority_row = (
    stock_recommendations_df
    .sort_values("priority_score", ascending=False)
    .iloc[0]
)

total_purchase_units = int(
    stock_recommendations_df["recommended_purchase_quantity"].sum()
)

print("Products that need replenishment:", len(products_to_restock))
print("Critical products:", len(critical_products))
print("Highest-priority product:", highest_priority_row["product_name"])
print("Total units recommended:", total_purchase_units)

Products that need replenishment: 0
Critical products: 0
Highest-priority product: Capa de Chuva
Total units recommended: 0


Products without an available stock snapshot were filled with 0 as a conservative assumption.


In [23]:
status_counts = stock_recommendations_df["stock_status"].value_counts().reindex(
    ["critical", "warning", "healthy", "overstock"], fill_value=0
)
conclusion_metrics = {
    "products_requiring_replenishment": int((stock_recommendations_df["recommended_purchase_quantity"] > 0).sum()),
    "critical_products": int(status_counts["critical"]),
    "highest_priority_product": str(stock_recommendations_df.iloc[0]["product_name"]) if not stock_recommendations_df.empty else None,
    "total_recommended_units": int(stock_recommendations_df["recommended_purchase_quantity"].sum()),
}
print(conclusion_metrics)
print("Stock status counts:")
print(status_counts)

# The saved summary must reconcile with the detailed report.
assert int(recommendation_summary_df.loc[0, "total_products"]) == len(stock_recommendations_df)
assert int(recommendation_summary_df.loc[0, "total_recommended_purchase_units"]) == conclusion_metrics["total_recommended_units"]
assert int(recommendation_summary_df.loc[0, "critical_products"]) == conclusion_metrics["critical_products"]


{'products_requiring_replenishment': 0, 'critical_products': 0, 'highest_priority_product': 'Capa de Chuva', 'total_recommended_units': 0}
Stock status counts:
stock_status
critical     0
warning      0
healthy      3
overstock    9
Name: count, dtype: int64


## Stock Recommendation Conclusion

The stock recommendation engine used the selected XGBoost model to forecast demand for the next 14 days. The code cell above shows the actual number of products requiring replenishment, critical products, highest-priority product and total recommended units.

The daily predictions were aggregated by product and combined with current stock, safety stock and supplier lead time. Products classified as critical should receive attention first because their available stock is not enough to cover forecasted demand. The priority score orders products according to stock risk, purchase quantity and supplier lead time.

Safety stock adds a 20% margin to reduce stockout risk from demand variation, forecast errors and supplier delays.

### Current Limitations

- A fixed supplier lead time of 7 days is currently used.
- Safety stock is calculated using a fixed rule of 20% of forecasted demand.
- Current stock is based on the latest available valid snapshot and is not real-time inventory.
- Products without an available stock snapshot are filled with 0 as a conservative assumption.
- Future lag and rolling features are based on the latest historical product state.
- The recommendations should be reviewed before real purchase decisions.
